In [ ]:
from pathlib import Path
from matplotlib import pyplot as plt
from vascx.fundus.loader import RetinaLoader
from vascx.utils.analysis import (
    extract_in_parallel,
)

In [ ]:
ds_path = Path("../../samples/fundus")

In [ ]:
loader = RetinaLoader.from_folder(ds_path)

In [ ]:
r = loader[1]

In [ ]:
segments = [seg for seg in r.veins.segments if seg.length > 100]

In [ ]:
seg = segments[11]

In [ ]:
seg.plot()

In [ ]:
profile = seg.profile
plt.imshow(profile, cmap='gray')

In [ ]:
from random import sample
import numpy as np
from scipy.stats import norm
from scipy.optimize import curve_fit
from scipy.optimize import least_squares
from rtnls_enface.utils.plotting import mpl_grid

In [ ]:
x= np.arange(-5,5)
y= np.arange(-10,10)
C = 150
A = 10
mx=1
mu = 0
sigma=1

In [ ]:
x,y

In [ ]:
r = C + mx * x[:,None] - A*(np.exp(-0.5 * ((y[None,:] - mu) / sigma) ** 2) )

In [ ]:
(np.exp(-0.5 * ((y - mu) / sigma) ** 2) ).shape

In [ ]:
r.shape

In [ ]:
def gaussian_2d(x, y, C, m, A, mu, sigma):
    return C + m * x[:,None] - A*(np.exp(-0.5 * ((y[None,:] - mu) / sigma) ** 2) )

# Objective function for least_squares with weights
def residuals_2d(params, x, y, D, weights):
    D_model = gaussian_2d(x, y, *params)
    residual = (D - D_model) * weights[:,None]  # Apply spatial weights to the residuals
    return residual.ravel()  # Flatten the residual to a 1D array

In [ ]:
def fit_vessel_2d_gaussian(segment):
    D = segment.profile
    x = np.linspace(-D.shape[0] // 2 + 1, D.shape[0] // 2, D.shape[0])
    y = np.linspace(-D.shape[1] // 2 + 1, D.shape[1] // 2, D.shape[1])
    weights = np.ones_like(x)
    initial_guess = [np.max(D), 0, np.max(D) - np.min(D), 0, segment.median_diameter]
    result = least_squares(residuals_2d, initial_guess, args=(x,y,D, weights))
    return result.x

In [ ]:
def plot_profiles_1d(segment):
    optimized_params = fit_vessel_2d_gaussian(segment)
    y = np.linspace(-segment.profile.shape[1] // 2 + 1, segment.profile.shape[1] // 2, segment.profile.shape[1])

    ids = sample(range(len(segment.profile)), 8)
    xs = np.array(ids) - len(segment.profile) // 2 + 1
    slices = segment.profile[ids,:]
    with mpl_grid(slices.shape[0] // 4, 4, figsize=(10,4)) as (fig, axes):
        for i, ax in enumerate(axes):
            Y = slices[i,:]

            x = xs[i][None]
            ax.plot(y, Y)
            # ax.plot(y, gaussian_2d(x, y , *initial_guess)[0])
            ax.plot(y, gaussian_2d(x, y , *optimized_params)[0])
            # ax.plot(x, [gaussian_2d(x_i, *popt) for x_i in x])
            # ax.axvline(x=mu-sigma, color='r', linestyle='--', linewidth=2)
            # ax.axvline(x=mu+sigma, color='r', linestyle='--', linewidth=2)

In [ ]:
def plot_profile(segment):
    plt.imshow(segment.profile, cmap='gray')
    optimized_params = fit_vessel_2d_gaussian(segment)
    C, m, A, mu, sigma = optimized_params
    print(mu, sigma, segment.profile.shape)
    plt.axvline(x=segment.profile.shape[1]//2+mu-sigma, color='r', linestyle='--', linewidth=2)
    plt.axvline(x=segment.profile.shape[1]//2+mu+sigma, color='r', linestyle='--', linewidth=2)

In [ ]:
segment = segments[13]

In [ ]:
segment.plot()

In [ ]:
plot_profile(segment)

In [ ]:
plot_profiles_1d(segments[6])

In [ ]:
plot_profile()

In [ ]:
def gaussian_with_offset(x, A, mu, sigma, c):
    return c - A * (np.exp(-0.5 * ((x - mu) / sigma) ** 2) )
def fit_gaussian(y):
    # Define a Gaussian model with an offset c
    x = np.linspace(-len(y) // 2 + 1, len(y) // 2, len(y))
    init = [np.max(y) - np.min(y), 0, seg.median_diameter, np.max(y)]
    popt, _ = curve_fit(gaussian_with_offset, x, y, p0=init)

    return init, popt



In [ ]:
plt.imshow(profile, cmap='binary')